# R2 · 7 · Gráficos por UF: semanal e temporada com os campeões de cada área

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas anteriores, inclusive `semanal_lib.py`):
1. suba **`r2_lib.py`** (pasta `data_imdc_2026/RODADA2/`);
2. **crie à mão** as pastas `resultados_semanal_r2`, `resultados_uf_r2`, `cache_r2` e `avaliacao_r2` (evita pastas duplicadas quando há duas sessões abertas).

**Nada antigo é alterado.** Os modelos novos gravam em `resultados_semanal_r2/` e `resultados_uf_r2/`; as tabelas novas vão para `avaliacao_r2/` com a data e a hora no nome (`..._AAAAMMDD-HHMM.csv`) e uma cópia `..._ultimo.csv`. As tabelas da rodada anterior (`resultados_semanal_avaliacao/`) continuam como estão.

**Retomada:** se a internet ou a energia cair, reconecte e **rode tudo de novo**. Cada modelo × rodada/temporada vira um arquivo assim que termina (gravado como `.tmp` e renomeado); os treinos longos das redes gravam um checkpoint a cada 5 épocas e continuam da última época salva; o tsfresh grava uma UF por vez; o Optuna grava o estudo no Drive a cada tentativa; o Chronos ajustado fica salvo e não é refeito.

CPU comum, ~10–20 min. **Rode depois do R2_1 final** — este notebook só lê previsões e os arquivos que o R2_1 gravou em `avaliacao_r2/` (λ da temporada, calibração e os pesos dos ensembles com pesos) e recria exatamente os mesmos ensembles. Nada é treinado de novo.

**Em cada figura (uma por UF e uma para o Brasil):**
- **em cima, semanal (teste 2022–2025):** casos observados (preto), previsão de 1 semana à frente do melhor modelo semanal (**ENS_GBM_ARIMA_MTGNNcresc_sem**, azul, com a faixa de 90%) e previsão de 4 semanas à frente do melhor em 4 semanas (**ENS_stack_sem**, laranja tracejado);
- **embaixo, temporada:** previsão feita em junho (SE25) para a temporada SE41→SE40 seguinte, nas 4 temporadas de teste: o melhor geral (**ENS_grafos3_λ**, azul com faixas de 50% e 90%), o melhor no pico entre os nossos (**ENS_stack_UF**, vermelho), o melhor para o total da temporada (**MTGNN_total_pp_UF+cal**, verde pontilhado) e o modelo-base oficial (**BB**, cinza tracejado).

O Brasil é a soma das UFs (as faixas somadas quantil a quantil ficam um pouco largas demais). Saída: `avaliacao_r2/graficos_r2_<carimbo>/` com um PNG por UF, um PDF com todas e `totais_temporada.csv` (total observado × previsto por UF e temporada). Retomada: os PNGs prontos são pulados.


In [ ]:
NOTEBOOK = 'r2_graficos'
PACOTES = ''

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r2', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'dados', 'dados_uf',
                 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_SEM_R2 = os.path.join(PASTA_DRIVE, 'resultados_semanal_r2')
PASTA_UF_R2 = os.path.join(PASTA_DRIVE, 'resultados_uf_r2')
PASTA_CACHE = os.path.join(PASTA_DRIVE, 'cache_r2')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
ARQ_SEL = os.path.join(PASTA_DRIVE, 'resultados_semanal', 'selecao_grupos.json')
for p in (PASTA_SEM_R2, PASTA_UF_R2, PASTA_CACHE, PASTA_AV2, os.path.join(PASTA_CACHE, 'ckpt')):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2
for m in (V, L, CM, U, S, R2):
    importlib.reload(m)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
except ImportError:
    torch = None
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
RODADAS = S.rodadas(D, felipe=True)
LIM = R2.limiares(D)                        # p25/p75 de cada UF com dados até 2017 (antes da afinação)
REFAZER = False


In [ ]:
# @title 2 · Semanal: membros, ensemble de mediana e ensemble com pesos (pesos gravados pelo R2_1)
PASTAS_SEM = sorted(p for p in glob.glob(os.path.join(PASTA_DRIVE, 'resultados_semanal*'))
                    if os.path.isdir(p) and not p.endswith('_avaliacao'))
arq_ps = sorted(glob.glob(os.path.join(PASTA_AV2, 'pesos_stack_semanal_*.json')))
assert arq_ps, 'Rode antes o R2_1 (ele grava os pesos do ENS_stack_sem).'
PESOS_SEM = R2.json_le(arq_ps[-1])
MEMB_SEM = list(next(iter(PESOS_SEM.values())).keys())
MED3 = ['GBM_xgb_todas_sem', 'ARIMA_Fourier_sem', 'MTGNN_cresc_sem']
PRECISA = set(MEMB_SEM) | set(MED3)
fs = [f for p in PASTAS_SEM for f in glob.glob(os.path.join(p, '*.parquet')) if os.path.basename(f).split('__')[0] in PRECISA]
P0 = pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True).drop_duplicates(['modelo', 'rodada', 'h', 'date', 'uf_code'], keep='last')
P0 = P0[P0.rodada.map(S.grupo_rodada) == 'teste']
P0['date'], P0['origem'] = pd.to_datetime(P0['date']), pd.to_datetime(P0['origem'])
E1 = S.ensemble_semanal(P0, MED3, 'ENS_GBM_ARIMA_MTGNNcresc_sem')
PV = S.com_verdade(P0, D)
PV['fase_origem'] = R2.fase_origem(PV, D, LIM)
PV['balde'] = PV.h.astype(str) + '|' + PV.fase_origem
ST = R2.aplicar_stacking(PV, MEMB_SEM, ['rodada', 'h', 'date', 'uf_code', 'balde'], 'balde', PESOS_SEM, 'ENS_stack_sem')
SEM = {'1 semana à frente · ENS_GBM_ARIMA_MTGNNcresc_sem': E1[E1.h == 1].assign(date=lambda d: pd.to_datetime(d.date)),
       '4 semanas à frente · ENS_stack_sem': ST[ST.h == 4].assign(date=lambda d: pd.to_datetime(d.date))}
print('membros do stacking semanal:', MEMB_SEM, '·', {k: len(v) for k, v in SEM.items()})


In [ ]:
# @title 3 · Temporada: mesmas fontes, λ, ensembles e calibração do R2_1
PASTAS_UF = [p for p in sorted(glob.glob(os.path.join(PASTA_DRIVE, 'resultados_uf*'))) if os.path.isdir(p)]
FONTES = [('região→UF', PASTA_COMP)] + [(os.path.basename(p), p) for p in PASTAS_UF] + [('externo', PASTA_EXT)]
PU = L.carregar_previsoes(FONTES, ctx.Du.ufs, 'uf', mostrar=False)
PU = PU[~PU.modelo.str.startswith('ENS')][L.COLS_PREV]
ens0 = [os.path.join(p, 'ensembles_uf.parquet') for p in PASTAS_UF if os.path.exists(os.path.join(p, 'ensembles_uf.parquet'))]
ENS0 = pd.read_parquet(ens0[0])[L.COLS_PREV] if ens0 else PU.iloc[:0]
REF = ctx.ref[L.COLS_PREV] if ctx.ref is not None else PU.iloc[:0]
LAM_T = R2.json_le(os.path.join(PASTA_AV2, 'lambda_temporada.json'), {})
CAL2 = [U.ajusta_largura(PU[PU.modelo == m], l).assign(modelo=f'{m}+λ') for m, l in LAM_T.items() if l != 1.0 and (PU.modelo == m).any()]
TODOS = pd.concat([REF, PU, ENS0] + CAL2, ignore_index=True).drop_duplicates(['modelo', 'season', 'uf_code', 'date'], keep='last')
TODOS = TODOS[TODOS.season.isin(TESTE)]
arq_pt = sorted(glob.glob(os.path.join(PASTA_AV2, 'pesos_stack_temporada_*.json')))
assert arq_pt, 'Rode antes o R2_1 (ele grava os pesos do ENS_stack_UF).'
PESOS_T = R2.json_le(arq_pt[-1])
MEMB_T = list(next(iter(PESOS_T.values())).keys())
BASE_G = ['MTGNN_UF+λ', 'A3TGCN_UF+λ', 'GCNGRU_UF+λ']
FIXOS = ['MTGNN_pesopop_UF+λ', 'TFT_pool_UF', 'DoisEstagios_pool_UF'] + BASE_G
b_todos = [m for m in MEMB_T if m not in FIXOS]                  # o melhor modelo novo na afinação (o R2_1 o põe nos dois)
e3, _ = V.ensemble(TODOS, BASE_G + b_todos[:1], 'ENS_grafos3_λ', metodo='mediana')
c_data = {s: pd.Timestamp(D.datas[D.corte(s)]) for s in TESTE}
TV = S.com_verdade(TODOS, D)
TV['h'] = ((TV.date - TV.season.map(c_data)).dt.days // 7).astype(int)
TV['balde'] = R2.bloco_horizonte(TV.h.values)
STT = R2.aplicar_stacking(TV, MEMB_T, ['season', 'date', 'uf_code', 'balde'], 'balde', PESOS_T, 'ENS_stack_UF')
CAL_T = R2.json_le(os.path.join(PASTA_AV2, 'calibracao_temporada.json'), {})
TOT = R2.aplicar_cal(TV[TV.modelo == 'MTGNN_total_pp_UF'], 'balde', CAL_T['MTGNN_total_pp_UF']) if 'MTGNN_total_pp_UF' in CAL_T else TV.iloc[:0]
data = lambda d: d.assign(date=pd.to_datetime(d.date))
TEMP = {'melhor geral · ENS_grafos3_λ': data(e3), 'melhor no pico · ENS_stack_UF': data(STT),
        'melhor no total · MTGNN_total_pp_UF+cal': data(TOT), 'modelo-base oficial · BB': data(TODOS[TODOS.modelo == 'BB'])}
print('novo do ensemble:', b_todos, '·', {k: len(v) for k, v in TEMP.items()})


In [ ]:
# @title 4 · Figuras (uma por UF + Brasil), PDF único e totais por temporada
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import Image, display
PASTA_G = os.path.join(PASTA_AV2, f'graficos_r2_{R2.carimbo()[:8]}')
os.makedirs(PASTA_G, exist_ok=True)
OBS = pd.DataFrame(D.Y[:D.ultimo_obs + 1], index=pd.to_datetime(D.datas[:D.ultimo_obs + 1]), columns=D.unidades)
OBS[0] = OBS.sum(1)
ORDEM = [0] + sorted(D.unidades, key=lambda u: R2.SIGLAS.get(u, str(u)))
tot = []
for u in ORDEM:
    arq = os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png')
    if not os.path.exists(arq) or REFAZER:
        fig = R2.grafico_uf(OBS[u], SEM, TEMP, u, arq, TESTE, ' · Brasil = soma das UFs' if u == 0 else '')
        plt.close(fig)
    tot.append(R2.totais_temporada(OBS[u], TEMP, u, TESTE))
    print('✅', R2.SIGLAS.get(u, u), flush=True)
with PdfPages(os.path.join(PASTA_G, 'graficos_UFs_r2.pdf.tmp')) as pdf:
    for u in ORDEM:
        img = plt.imread(os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png'))
        f = plt.figure(figsize=(13, 8.2)); plt.imshow(img); plt.axis('off'); pdf.savefig(f, bbox_inches='tight'); plt.close(f)
os.replace(os.path.join(PASTA_G, 'graficos_UFs_r2.pdf.tmp'), os.path.join(PASTA_G, 'graficos_UFs_r2.pdf'))
TOT_T = pd.concat(tot, ignore_index=True)
TOT_T.to_csv(os.path.join(PASTA_G, 'totais_temporada.csv'), index=False)
print('Pasta:', PASTA_G)
for u in (0, 35, 31, 41, 52):
    display(Image(os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png')))
TOT_T.round(0)
